In [1]:
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_score,
    recall_score,
    roc_auc_score,
)

train = pd.read_csv("train.csv")

target_col = "is_fraud"
id_col = "id"

# 1. Danh sách các cột categorical chính xác theo model.txt
cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]

# 2. Loại bỏ cả target_col và id_col khỏi feature_cols
feature_cols = [col for col in train.columns if col not in [target_col, id_col]]

X = train[feature_cols].copy()
y = train[target_col]

# 3. Chuyển đổi các cột text/object thành kiểu category
for col in cat_cols:
    X[col] = X[col].astype("category")

In [2]:
import lightgbm as lgb

booster = lgb.Booster(model_file="model.txt")
print(f"Tổng số cây: {booster.num_trees()}")

# Dự đoán xác suất (predict_proba)
prob = booster.predict(X)

# Ngưỡng phân loại (threshold)
threshold = 0.5
pred = (prob >= threshold).astype(int)

# Đánh giá 4 chỉ số
roc_auc = roc_auc_score(y, prob)
auc_pr = average_precision_score(y, prob)
precision = precision_score(y, pred, zero_division=0)
recall = recall_score(y, pred, zero_division=0)

print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"AUC-PR    : {auc_pr:.4f}")
print(f"Precision : {precision:.4f} (với threshold = {threshold})")
print(f"Recall    : {recall:.4f} (với threshold = {threshold})")

print("\nBáo cáo chi tiết:")
print(classification_report(y, pred, digits=4))

Tổng số cây: 1484
ROC-AUC   : 0.9516
AUC-PR    : 0.5349
Precision : 0.7735 (với threshold = 0.5)
Recall    : 0.3240 (với threshold = 0.5)

Báo cáo chi tiết:
              precision    recall  f1-score   support

         0.0     0.9886    0.9984    0.9935    688271
         1.0     0.7735    0.3240    0.4567     11689

    accuracy                         0.9871    699960
   macro avg     0.8811    0.6612    0.7251    699960
weighted avg     0.9850    0.9871    0.9845    699960



In [3]:
# Dự đoán trên test.csv và xuất submission cho LightGBM
try:
    test = pd.read_csv("test.csv")
    X_test = test[feature_cols].copy()
    
    # Chuyển đổi các cột phân loại sang kiểu 'category' tương tự như tập train
    for col in cat_cols:
        X_test[col] = X_test[col].astype("category")
    
    # Booster của LightGBM dự đoán trực tiếp ra xác suất (predict_proba)
    test_prob = booster.predict(X_test)
    
    submission = pd.DataFrame({
        id_col: test[id_col],
        target_col: test_prob
    })
    submission.to_csv("submission_lgb.csv", index=False)
    print("Đã lưu kết quả dự đoán vào submission_lgb.csv thành công!")
except Exception as e:
    print("Lỗi dự đoán test:", e)
    raise e

Đã lưu kết quả dự đoán vào submission_lgb.csv thành công!
